# Part 3: Clinical Summarization

This notebook implements Part 3 of the in-class Hugging Face activity. It runs a summarization pipeline on the provided messy clinical note and then organizes a critical review of the generated summary.

In [8]:
import torch
from textwrap import fill

from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, pipeline

### Source Note

In [9]:
note = """
ED triage: 67-year-old male with HTN and type 2 diabetes presents with
chest pressure radiating to left arm for 2 hours after mowing lawn.
Reports nausea and diaphoresis. Home meds listed as lisinopril and metformin.
Allergies: NKDA.

Nursing addendum 15 min later: patient states he is 64, not 67. Says pain
started yesterday, not today, and now denies radiation, nausea, or sweating.
States he stopped taking all meds 3 months ago. Allergy listed as penicillin
causes rash.

Resident note: wife says he nearly passed out in driveway and was clutching
his chest. Patient denies diabetes but prior chart shows A1c 9.1 last month.
Initial EKG documented as ST elevation in II, III, aVF; repeat note says
"no acute ST changes." Troponin pending.
""".strip()

print(note)

ED triage: 67-year-old male with HTN and type 2 diabetes presents with
chest pressure radiating to left arm for 2 hours after mowing lawn.
Reports nausea and diaphoresis. Home meds listed as lisinopril and metformin.
Allergies: NKDA.

Nursing addendum 15 min later: patient states he is 64, not 67. Says pain
started yesterday, not today, and now denies radiation, nausea, or sweating.
States he stopped taking all meds 3 months ago. Allergy listed as penicillin
causes rash.

Resident note: wife says he nearly passed out in driveway and was clutching
his chest. Patient denies diabetes but prior chart shows A1c 9.1 last month.
Initial EKG documented as ST elevation in II, III, aVF; repeat note says
"no acute ST changes." Troponin pending.


### Run the Summarization Pipeline.

In [10]:
model_id = "facebook/bart-large-cnn"

# These calls download the tokenizer/model the first time you run them, then load from cache later.
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForSeq2SeqLM.from_pretrained(model_id, use_safetensors=True)

print(f"Loaded summarization model: {model_id}")

Loaded summarization model: facebook/bart-large-cnn


In [11]:
summarizer = pipeline("summarization", model=model, tokenizer=tokenizer, framework="pt")

print(summarizer(note, max_length=100))

Device set to use cuda:0


[{'summary_text': '67-year-old male with HTN and type 2 diabetes presents with chest pressure radiating to left arm for 2 hours after mowing lawn. Reports nausea and diaphoresis. Home meds listed as lisinopril and metformin. Patient denies diabetes but prior chart shows A1c 9.1 last month.'}]


In [12]:
summary_result = summarizer(note, max_length=40, min_length=12, do_sample=False)
summary_text = summary_result[0]["summary_text"]

print("Generated summary:\n")
print(fill(summary_text, width=100))

Generated summary:

Patient denies diabetes but prior chart shows A1c 9.1 last month. Wife says he nearly passed out in
driveway and was clutching his chest.


## Analysis

First, we note that the unsummarized text already contained inconsistencies, such as the patients age. This makes it difficult for the model to come up with an accuracte summary. 